In [2]:
:dep smartcore = { version = "0.3.2", features = ["ndarray-bindings", "serde"] }
:dep bincode = "1.3"
:dep csv = "1.1"
:dep ndarray = "0.15" 
:dep linfa = { version = "0.7", features = ["default"] }
:dep rand = "0.8"
:dep rand_chacha = "0.3"
:dep plotters = "0.3"
:dep linfa-preprocessing = "0.7"
:dep linfa-elasticnet = "0.7"

In [34]:
use smartcore::{neighbors::knn_classifier::*,linalg::basic::matrix::DenseMatrix,
                naive_bayes::gaussian::GaussianNB,tree::decision_tree_classifier::DecisionTreeClassifier,
                metrics::distance::Distances,tree::decision_tree_classifier::DecisionTreeClassifierParameters,
                tree::decision_tree_classifier::SplitCriterion};
use csv;
use linfa::{Dataset};
use ndarray::{Array1, Array2, ArrayView1, ArrayView2, Axis, Ix1, array,s};
use rand::SeedableRng;
use rand_chacha::ChaCha12Rng;
use linfa_preprocessing::linear_scaling::LinearScaler;
use linfa::traits::{Fit, Transformer};
use linfa_elasticnet::{ElasticNet};

In [35]:
let rec_path= "train_n.csv";
let targ_path= "train_n_y.csv";
let mut reader_rec =csv::Reader::from_path(rec_path).unwrap();
let mut reader_targ =csv::Reader::from_path(targ_path).unwrap();
let mut records: Array2<f64> = Array2::zeros((0, 22));  //X
let mut targets: Array1<f64>= Array1::zeros(0); //Y
let mut rng = ChaCha12Rng::seed_from_u64(5);

In [36]:
for data in reader_rec.records(){
        let rec = data.unwrap();
        let row_vec: Vec<f64> = (0..22)
        .map(|i| rec[i].parse::<f64>().expect("pupupu"))
        .collect();
        let array = Array2::from_shape_vec((1, 22), row_vec).expect("pupep");
        let _  = records.append(Axis(0), array.view());
}
for data in reader_targ.records(){
        let targ = data.unwrap();
        let _ = targets.append(Axis(0),array![targ[0].parse::<f64>().expect("pupupu")].view());
}
let data_set= Dataset::new(records.clone(), targets.clone());
let (train_data,test_data) = data_set.shuffle(&mut rng)
                                     .split_with_ratio(0.85);

In [37]:
let scaler: LinearScaler<f64> = LinearScaler::standard().fit(&data_set).expect("fail to fit");
let scale_dataset = scaler.transform(data_set);
let model:ElasticNet<f64> = ElasticNet::params()
    .l1_ratio(1.0)   
    .penalty(0.1)    
    .fit(&scale_dataset)
    .expect("fail to fit Lasso");